# Lab 1 — Agentic project (date MCP)

Make your own MCP server with a date tool, then let an Agent use it.

- Build/verify `date_server.py` (`get_current_date`)
- Connect via `MCPServerStdio`
- Run an `Agent` + `Runner`
- Optional: use `date_client.py` + native OpenAI call (no Agents SDK)

Files used in this folder: `date_server.py`, `date_client.py`, `openai_mcp_bridge.py`.

### Load env and build date MCP params

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

EXERCISE_DIR = Path.cwd().resolve()
if not (EXERCISE_DIR / "date_server.py").exists():
    EXERCISE_DIR = Path("/home/kayode/Documents/AI/agents/6_mcp/community_contributions/adeyemi_kayode")

REPO_ROOT = EXERCISE_DIR.parent.parent.parent

date_mcp_params = {
    "command": "uv",
    "args": ["run", str(EXERCISE_DIR / "date_server.py")],
    "cwd": str(REPO_ROOT),
}

print("EXERCISE_DIR =", EXERCISE_DIR)
print("REPO_ROOT   =", REPO_ROOT)

EXERCISE_DIR = /home/kayode/Documents/AI/agents/6_mcp/community_contributions/adeyemi_kayode
REPO_ROOT   = /home/kayode/Documents/AI/agents


In [2]:
async with MCPServerStdio(params=date_mcp_params, client_session_timeout_seconds=30) as server:
    tools = await server.list_tools()

tools

[Tool(name='get_current_date', title=None, description="Return today's calendar date in ISO format (YYYY-MM-DD).", inputSchema={'properties': {}, 'title': 'get_current_dateArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_current_dateOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)]

### Run the Agent against your date tool

In [3]:
request = "What is today's date? Use the available tool."

async with MCPServerStdio(params=date_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(
        name="date_helper",
        instructions="Use tools and answer briefly.",
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
    )
    with trace("adeyemi_lab1_date_agent"):
        result = await Runner.run(agent, request)

print(result.final_output)

Today's date is March 31, 2026.


### Optional stretch

- Use `date_client.py` to call the tool without `Agent`.
- Use `openai_mcp_bridge.py` + native OpenAI Chat Completions tool loop (no Agents SDK).

Open `2_lab.ipynb` for the trader project.

### Optional native OpenAI (no Agents SDK)

Use `date_client.py` + `openai_mcp_bridge.py` to run a manual `tool_calls` loop if you want the harder exercise.

In [4]:
import sys

sys.path.insert(0, str(EXERCISE_DIR))
from date_client import list_date_tools

tools = await list_date_tools()
for t in tools:
    print(t.name)

get_current_date


In [5]:
# Optional harder exercise: native OpenAI call (no Agent/Runner)
import json

from openai import AsyncOpenAI
from date_client import date_server_session
from openai_mcp_bridge import mcp_tools_to_openai_functions, tool_result_text


async def native_openai_date() -> None:
    client = AsyncOpenAI()
    async with date_server_session() as session:
        listed = await session.list_tools()
        tools = mcp_tools_to_openai_functions(listed.tools)
        messages = [{"role": "user", "content": "What is today's date? Call the tool."}]

        for _ in range(5):
            completion = await client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                tools=tools,
            )
            msg = completion.choices[0].message
            if not msg.tool_calls:
                print(msg.content)
                return

            messages.append(msg.model_dump())
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments or "{}")
                res = await session.call_tool(tc.function.name, args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_result_text(res),
                })


await native_openai_date()

Today's date is March 31, 2026.


Open `2_lab.ipynb` for the trader project.